# Make the Chatbot signing key (one time)

This creates the **permanent signing key** for your Chatbot app. With it, every new version installs **over** the old app and **keeps your downloaded models, chats and keys**.

**How to use:** click **Runtime → Run all**. Then:
1. Type a password when asked (at least 6 characters). **Write it down somewhere safe.**
2. Allow Google Drive access. A backup of the key is saved to **My Drive → Chatbot → signing-key**.
3. Add the 3 secrets shown at the end to GitHub.

> ⚠️ **Never lose the key file or the password.** Without them, future updates can't install over the app.

In [ ]:
import os, subprocess, base64, getpass, shutil

# Java's keytool makes the key.
if shutil.which("keytool") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null",
                   shell=True, check=True)

while True:
    pw = getpass.getpass("Choose a password (min 6 characters): ")
    if len(pw) < 6:
        print("Too short, try again."); continue
    if getpass.getpass("Type it again: ") != pw:
        print("They don't match, try again."); continue
    break

if os.path.exists("upload-keystore.jks"):
    os.remove("upload-keystore.jks")
subprocess.run(["keytool", "-genkeypair", "-keystore", "upload-keystore.jks", "-alias", "upload",
                "-keyalg", "RSA", "-keysize", "2048", "-validity", "10000",
                "-storepass", pw, "-keypass", pw, "-dname", "CN=Chatbot, O=Robokks"],
               check=True, capture_output=True)
b64 = base64.b64encode(open("upload-keystore.jks", "rb").read()).decode()
print("✓ Key created.")


In [ ]:
# Back up to Google Drive: My Drive/Chatbot/signing-key
from google.colab import drive
drive.mount("/content/drive")
folder = "/content/drive/MyDrive/Chatbot/signing-key"
os.makedirs(folder, exist_ok=True)
shutil.copy("upload-keystore.jks", f"{folder}/upload-keystore.jks")
open(f"{folder}/keystore-base64.txt", "w").write(b64)
open(f"{folder}/README-keep-safe.txt", "w").write(
    "Chatbot app signing key. Keep this folder and your password safe.\n"
    "Key alias: upload\n"
    "The password is NOT stored here. Keep it in a password manager.\n"
    "GitHub secrets: ANDROID_KEYSTORE_BASE64 (keystore-base64.txt), "
    "ANDROID_KEYSTORE_PASSWORD (your password), ANDROID_KEY_ALIAS = upload\n")
print("✓ Backup saved to My Drive → Chatbot → signing-key")


## Now add 3 secrets on GitHub

Open **https://github.com/Robokks/ai-chatbot/settings/secrets/actions** and tap **New repository secret** three times:

| Name | Secret |
|---|---|
| `ANDROID_KEYSTORE_BASE64` | the long text printed below (also in **keystore-base64.txt** in your Drive folder, easier to copy on a phone) |
| `ANDROID_KEYSTORE_PASSWORD` | the password you chose |
| `ANDROID_KEY_ALIAS` | `upload` |

Then tell Claude it's done: it starts a new build signed with your key.

In [ ]:
print("ANDROID_KEYSTORE_BASE64 =")
print(b64)
print()
print("ANDROID_KEY_ALIAS = upload")
print("ANDROID_KEYSTORE_PASSWORD = (the password you typed)")
